In [10]:
import sys
import os
main_dir = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(main_dir)

import argparse
import os
import time

import numpy as np

from RemoteSensing.changedetection.configs.config import get_config

import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from RemoteSensing.changedetection.datasets.make_data_loader import SemanticChangeDetectionDatset, make_data_loader, SemanticChangeDetectionDatset_LandSat
from RemoteSensing.changedetection.utils_func.metrics import Evaluator
from RemoteSensing.changedetection.models.STMambaSCD import STMambaSCD
import RemoteSensing.changedetection.utils_func.lovasz_loss as L
from torch.optim.lr_scheduler import StepLR

from RemoteSensing.changedetection.utils_func.mcd_utils import accuracy, SCDD_eval_all, AverageMeter
import matplotlib.pyplot as plt

In [11]:
torch.cuda.set_device(0)

In [12]:
MODEL_PATH = os.path.join(
    main_dir, "RemoteSensing/saved_models/CA_spatial_fft_17_7LandSat/11500_model_0.305.pth"
)

In [13]:
from dotenv import load_dotenv

load_dotenv()

def getPath(env_path):
    return os.path.expanduser(os.getenv(env_path))

VSSM_MODEL_PATH = getPath('VSSMBASEPATH')

LandSat_DATASET_PATH = getPath('LANDSAT')
print(LandSat_DATASET_PATH)
LandSat_TRAIN_DATA_LIST_PATH = os.path.join(LandSat_DATASET_PATH, 'train_list.txt')
LandSat_TRAIN_DATA_LIST_PATH_2 = os.path.join(LandSat_DATASET_PATH, 'val_list.txt')

LandSat_TEST_DATA_LIST_PATH = os.path.join(LandSat_DATASET_PATH, 'test_list.txt')

train_data_list = []
with open(LandSat_TRAIN_DATA_LIST_PATH, 'r') as f:
    for line in f:
        train_data_list.append(line.strip())

with open(LandSat_TRAIN_DATA_LIST_PATH_2, 'r') as f:
    for line in f:
        train_data_list.append(line.strip())

test_data_list = []
with open(LandSat_TEST_DATA_LIST_PATH, 'r') as f:
    for line in f:
        test_data_list.append(line.strip())


configs_path = os.path.join(main_dir, 'RemoteSensing/changedetection/configs/vssm1/vssm_base_224.yaml')
model_path = os.path.join(main_dir, 'RemoteSensing/saved_models')

class ARGS:
    def __init__(self):
        self.cfg = configs_path
        self.opts = None
        self.pretrained_weight_path = None
        self.dataset = 'LandSat'
        self.type = 'train'
        self.train_dataset_path = LandSat_DATASET_PATH
        self.train_data_list_path = LandSat_TRAIN_DATA_LIST_PATH
        self.test_dataset_path = LandSat_DATASET_PATH
        self.test_data_list_path = LandSat_TEST_DATA_LIST_PATH
        self.shuffle = True
        self.batch_size = 1
        self.crop_size = 256
        self.train_data_name_list = train_data_list
        self.test_data_name_list = test_data_list
        self.start_iter = 0
        self.cuda = True
        self.max_iters = 1600000
        self.model_type = 'MambaSCD_base'
        self.model_param_path = model_path
        self.resume = MODEL_PATH
        self.learning_rate = 1e-4
        self.momentum = 0.9
        self.weight_decay = 5e-4

args = ARGS()

/storage/scratch3/buddhiw-change-detection/Datasets/Landsat-SCD/


In [27]:
config = get_config(args)

train_data_loader = make_data_loader(args)

deep_model = STMambaSCD(
        output_cd = 2, 
        output_clf = 7 if args.dataset == "SECOND" else 5,
        pretrained=args.pretrained_weight_path,
        patch_size=config.MODEL.VSSM.PATCH_SIZE, 
        in_chans=config.MODEL.VSSM.IN_CHANS, 
        num_classes=config.MODEL.NUM_CLASSES, 
        depths=config.MODEL.VSSM.DEPTHS, 
        dims=config.MODEL.VSSM.EMBED_DIM, 
        # ===================
        ssm_d_state=config.MODEL.VSSM.SSM_D_STATE,
        ssm_ratio=config.MODEL.VSSM.SSM_RATIO,
        ssm_rank_ratio=config.MODEL.VSSM.SSM_RANK_RATIO,
        ssm_dt_rank=("auto" if config.MODEL.VSSM.SSM_DT_RANK == "auto" else int(config.MODEL.VSSM.SSM_DT_RANK)),
        ssm_act_layer=config.MODEL.VSSM.SSM_ACT_LAYER,
        ssm_conv=config.MODEL.VSSM.SSM_CONV,
        ssm_conv_bias=config.MODEL.VSSM.SSM_CONV_BIAS,
        ssm_drop_rate=config.MODEL.VSSM.SSM_DROP_RATE,
        ssm_init=config.MODEL.VSSM.SSM_INIT,
        forward_type=config.MODEL.VSSM.SSM_FORWARDTYPE,
        # ===================
        mlp_ratio=config.MODEL.VSSM.MLP_RATIO,
        mlp_act_layer=config.MODEL.VSSM.MLP_ACT_LAYER,
        mlp_drop_rate=config.MODEL.VSSM.MLP_DROP_RATE,
        # ===================
        drop_path_rate=config.MODEL.DROP_PATH_RATE,
        patch_norm=config.MODEL.VSSM.PATCH_NORM,
        norm_layer=config.MODEL.VSSM.NORM_LAYER,
        downsample_version=config.MODEL.VSSM.DOWNSAMPLE,
        patchembed_version=config.MODEL.VSSM.PATCHEMBED,
        gmlp=config.MODEL.VSSM.GMLP,
        use_checkpoint=config.TRAIN.USE_CHECKPOINT,
        ) 

checkpoint = torch.load(args.resume)
model_dict = {}
state_dict = deep_model.state_dict()
for k, v in checkpoint.items():
    if k in state_dict:
        model_dict[k] = v
state_dict.update(model_dict)
deep_model.load_state_dict(state_dict)


deep_model.cuda()
deep_model.eval()


=> merge config from /storage/scratch3/buddhiw-change-detection/ChangeDetection/CDMamba/RemoteSensing/changedetection/configs/vssm1/vssm_base_224.yaml
False


STMambaSCD(
  (encoder): Backbone_VSSM(
    (patch_embed): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): Permute()
      (2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (3): Permute()
      (4): GELU(approximate='none')
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (6): Permute()
      (7): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
    (layers): ModuleList(
      (0): Sequential(
        (blocks): Sequential(
          (0): VSSBlock(
            (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (op): SS2D(
              (out_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
              (in_proj): Linear(in_features=128, out_features=256, bias=False)
              (act): SiLU()
              (conv2d): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=256, bias=False)
              (out_proj): Linear(in_

In [28]:
dataset = None
    
if args.dataset == 'SECOND':
    dataset = SemanticChangeDetectionDatset(args.test_dataset_path, args.test_data_name_list, 256, None, 'test')

if args.dataset == 'LandSat':
    dataset = SemanticChangeDetectionDatset_LandSat(args.test_dataset_path, args.test_data_name_list, 256, None, 'test')

val_data_loader = DataLoader(dataset, batch_size=1, num_workers=4, drop_last=False)
torch.cuda.empty_cache()
acc_meter = AverageMeter()

preds_all = []
labels_all = []
with torch.no_grad():
    for itera, data in enumerate(val_data_loader):
        pre_change_imgs, post_change_imgs, labels_cd, labels_clf_t1, labels_clf_t2, _ = data

        pre_change_imgs = pre_change_imgs.cuda()
        post_change_imgs = post_change_imgs.cuda()
        labels_cd = labels_cd.cuda().long()
        labels_clf_t1 = labels_clf_t1.cuda().long()
        labels_clf_t2 = labels_clf_t2.cuda().long()


        # input_data = torch.cat([pre_change_imgs, post_change_imgs], dim=1)
        output_1, output_semantic_t1, output_semantic_t2 = deep_model(pre_change_imgs, post_change_imgs)

        labels_cd = labels_cd.cpu().numpy()
        labels_A = labels_clf_t1.cpu().numpy()
        labels_B = labels_clf_t2.cpu().numpy()

        outputs_A = output_semantic_t1.cpu().detach()
        outputs_B = output_semantic_t2.cpu().detach()
        change_mask = F.sigmoid(output_1).cpu().detach() > 0.5
        preds_A = torch.argmax(outputs_A, dim=1)
        preds_B = torch.argmax(outputs_B, dim=1)
        preds_A = (preds_A * change_mask.squeeze().long()).numpy()
        preds_B = (preds_B * change_mask.squeeze().long()).numpy()

        if itera % 100 == 0:
            print(f'iter is {itera}')

        for (pred_A, pred_B, label_A, label_B) in zip(preds_A, preds_B, labels_A, labels_B):
            acc_A, valid_sum_A = accuracy(pred_A, label_A)
            acc_B, valid_sum_B = accuracy(pred_B, label_B)
            preds_all.append(pred_A)
            preds_all.append(pred_B)
            labels_all.append(label_A)
            labels_all.append(label_B)
            acc = (acc_A + acc_B) * 0.5
            acc_meter.update(acc)

kappa_n0, Fscd, IoU_mean, Sek = SCDD_eval_all(preds_all, labels_all, 7 if args.dataset == "SECOND" else 5)
print(f'Kappa coefficient rate is {kappa_n0}, F1 is {Fscd}, OA is {acc_meter.avg}, '
        f'mIoU is {IoU_mean}, SeK is {Sek}')

    

iter is 0
iter is 100
iter is 200
iter is 300
iter is 400
Kappa coefficient rate is -0.17880691337499532, F1 is 0.07451063980045226, OA is 0.0850954778734509, mIoU is 0.04975148294495464, SeK is -0.06910153282469625
